# Using LLM to label unlabeled data for finetuning a model 


In [1]:
from elasticsearch import Elasticsearch
from dotenv import load_dotenv
import os
load_dotenv(".env")
import spacy
nlp = spacy.load("en_core_web_trf")

In [ ]:
# Configuration
TARGET_URI = os.getenv("ELASTIC_ENDPOINT")
API_KEY = os.getenv("ELASTIC_API_KEY")
INDEX_NAME = "ibfd_data-dockerindex_2026-03-03"

es = Elasticsearch(TARGET_URI, api_key=API_KEY)

# Test connection
print(es.info())

# Test index
if es.indices.exists(index=INDEX_NAME):
    print(f"Index '{INDEX_NAME}' exists")
    print(es.count(index=INDEX_NAME))
else:
    print(f"Index '{INDEX_NAME}' not found")

print("*******")

def search(index=INDEX_NAME, prefix="label", size=30):
    results = es.search( #try es.scan instead to retrieve all hits
        index=index,
        # query={"match": {"Text": f"{prefix}"}},
        # query={"_source": {"docid": f"{prefix}"}}, #prefix should be f stringed into here actually
        # query={"wildcard": {"docid": "*ipa*"}},
        query = {     "term": {         "Fields.collection_code": f"{prefix}"     } }
,

        size=size,
    )

    hits = results["hits"]["hits"]
    print(f"Found {len(hits)} docs with '{prefix}' in collection_code:\n")

    for i, hit in enumerate(hits, 1):
        print(f"--- [{i}] id={hit['_id']} ---")
        for key, val in hit["_source"].items():
            print(f"  {key}: {val}")
        print()

    return hits

In [ ]:
counter = 0
number_list = list()
import json
all_collections = list()
docids_list = list()
countries_list = list()
countries_per_list = list()
with open("recovered_docs.json", "r", encoding="utf-8") as f:
    json_data = json.load(f)
    parsed_dict = json_data["files"]#["file"]["blocks"]["collection"]
    uniq_dict = json_data["unique_docs"]
    for entry in parsed_dict:
        blocks = ( entry["blocks"])
        for ent in blocks:
            all_docids = ent["doc_id"]
            docids_list.append(all_docids)
            country_temp = ent["country"]
            if country_temp:
                countries_per_list.append(country_temp)
                for country_name in country_temp:
                    countries_list.append(country_name)
            for element in ent["collection"]:
                collnaam = element
                all_collections.append(collnaam)
    for dictionarie in uniq_dict:
        more_docids = dictionarie["doc_id"]
        docids_list.append(more_docids)
        more_collections = dictionarie["collection"]
        for colls in more_collections:
            all_collections.append(colls)
        more_countries = dictionarie["country"]
        if more_countries:
            countries_per_list.append(more_countries)
            for country in more_countries:
                countries_list.append(country)
from collections import Counter
Counter(all_collections)

Counter({'ecjd': 181,
         'ttcls': 76,
         'ecji': 11,
         'tns': 9,
         'et': 7,
         'irs': 3,
         'hrte': 3,
         'wht': 3,
         'ivm': 3,
         'bit': 2,
         'gttc2': 2})

In [3]:

coll_dict = {'bit',
 'ecjd',
 'ecji',
 'et',
 'gttc2',
 'hrte',
 'irs',
 'ivm',
 'tns',
 'ttcls',
 'wht'}


for entry in coll_dict:
    query = {     "term": {         "Fields.collection_code": f'{entry}'    } }
# query = {     "term": {         "Fields.collection_code": "irs"     } }

# # Sanity check
    total = es.count(index=INDEX_NAME, query=query)["count"]
    print(f"Matching docs for {entry}: {total:,}")
 
 ## ecjd, ipa, dcl, ttcls

Matching docs for irs: 2,766
Matching docs for ecji: 4,431
Matching docs for hrte: 34
Matching docs for ivm: 1,645
Matching docs for ecjd: 1,246
Matching docs for tns: 122,960
Matching docs for gttc2: 911
Matching docs for bit: 2,066
Matching docs for et: 2,882
Matching docs for wht: 205
Matching docs for ttcls: 13,224


In [ ]:
# ivm_hits = search(index=INDEX_NAME, prefix = "ivm", size=20) #3 used to be 20
# ecjd_hits = search(index=INDEX_NAME, prefix = "ecjd", size=50) #181
# et_hits = search(index=INDEX_NAME, prefix = "et", size=20) #7
# irs_hits = search(index=INDEX_NAME, prefix = "irs", size=10) #3
# hrte_hits = search(index=INDEX_NAME, prefix = "hrte", size=1500) #3
# gttc2_hits = search(index=INDEX_NAME, prefix = "gttc2", size=10) #2
# ecji_hits = search(index=INDEX_NAME, prefix = "ecji", size=10) #11
# bit_hits = search(index=INDEX_NAME, prefix = "bit", size=50) #2
# wht_hits = search(index=INDEX_NAME, prefix = "wht", size=10) #3
# tns_hits = search(index=INDEX_NAME, prefix = "tns", size=80) #9
# ttcls_hits = search(index=INDEX_NAME, prefix = "ttcls", size=150) #76

ivm_hits = search(index=INDEX_NAME, prefix = "ivm", size=1000) #3 used to be 20
ecjd_hits = search(index=INDEX_NAME, prefix = "ecjd", size=1000) #181
et_hits = search(index=INDEX_NAME, prefix = "et", size=1000) #7
irs_hits = search(index=INDEX_NAME, prefix = "irs", size=1000) #3
hrte_hits = search(index=INDEX_NAME, prefix = "hrte", size=1000) #3
gttc2_hits = search(index=INDEX_NAME, prefix = "gttc2", size=1000) #2
ecji_hits = search(index=INDEX_NAME, prefix = "ecji", size=1000) #11
bit_hits = search(index=INDEX_NAME, prefix = "bit", size=1000) #2
wht_hits = search(index=INDEX_NAME, prefix = "wht", size=1000) #3
tns_hits = search(index=INDEX_NAME, prefix = "tns", size=1000) #9
ttcls_hits = search(index=INDEX_NAME, prefix = "ttcls", size=1000) #76

In [ ]:
results = es.search( #try es.scan instead to retrieve all hits
        index=INDEX_NAME,
        query={"match": {"Text": f"ipa"}}
        # query={"_source": {"docid": f"{prefix}"}}, #prefix should be f stringed into here actually
        # query={"wildcard": {"docid": "*ipa*"}},
        # query = {     "term": {         "Fields.collection_code": f"{prefix}"     } }
,

        size=10,
    )

In [ ]:
print(results)

In [ ]:
#plan is to use ES - "_id" to corroborate with recovered.json's _doc_id
# then, if the ES id is not in the test set, we take sentences from there

In [ ]:
# query = {
#     "nested": {
#         "path": "Fields",
#         "query": {
#             "wildcard": {
#                 "Fields.docid": {
#                     "value": "*ipa*",
#                     "case_insensitive": True,
#                 }
#             }
#         },
#         "inner_hits": {"_source": ["Fields.docid"]},
#     }
# }

# resp = es.search(index=INDEX_NAME, query=query, size=10, source=["Text", "Fields.docid"])

# print(f"Total matches: {resp['hits']['total']['value']:,}\n")

# for i, doc in enumerate(resp["hits"]["hits"], 1):
#     inner = doc.get("inner_hits", {}).get("Fields", {}).get("hits", {}).get("hits", [])
#     matched = [h["_source"].get("docid") for h in inner]
#     text = doc["_source"].get("Text") or ""
#     print(f"[{i}] _id={doc['_id']}  docids={matched}  text_len={len(text):,}")
#     print(f"    {text[:200]!r}\n")

In [ ]:
# hits = search(INDEX_NAME,"ivm")

In [ ]:
counter = 0
number_list = list()
import json
all_collections = list()
docids_list = list()
countries_list = list()
countries_per_list = list()
with open("recovered_docs.json", "r", encoding="utf-8") as f:
    json_data = json.load(f)
    parsed_dict = json_data["files"]#["file"]["blocks"]["collection"]
    uniq_dict = json_data["unique_docs"]
    for entry in parsed_dict:
        blocks = ( entry["blocks"])
        for ent in blocks:
            all_docids = ent["doc_id"]
            docids_list.append(all_docids)
            country_temp = ent["country"]
            if country_temp:
                countries_per_list.append(country_temp)
                for country_name in country_temp:
                    countries_list.append(country_name)
            for element in ent["collection"]:
                collnaam = element
                all_collections.append(collnaam)
    for dictionarie in uniq_dict:
        more_docids = dictionarie["doc_id"]
        docids_list.append(more_docids)
        more_collections = dictionarie["collection"]
        for colls in more_collections:
            all_collections.append(colls)
        more_countries = dictionarie["country"]
        if more_countries:
            countries_per_list.append(more_countries)
            for country in more_countries:
                countries_list.append(country)

In [ ]:
# countries = list()
# es_doc_ids = list()
# print(hits[0])
# for hit in hits:
#     for keys in hit:
#         # print(keys)
#         da_id = hit["_id"]
#         da_countries = hit["_source"]["Fields"]["country"]
#     es_doc_ids.append(da_id)
#     countries.append(da_countries)


In [ ]:
def matches_in_hits(list_of_hits, all_docids, countries_list):
    matched = []
    for hit in list_of_hits:
        source = hit.get("_source", {})
        fields = source.get("Fields") or {}
        doc_id = hit.get("_id")
        country = fields.get("country")
        
        if doc_id in all_docids:
            continue
        if set(country or []) in [set(c) for c in countries_list]:
            matched.append(hit) 
    
    return matched

In [ ]:
def get_texts(matched):
    """"takes ibfd es dictionary and extracts the text and returns as one large string"""
    spacyfied_text_by_doc = []
    for match in matched:
        per_doc = []
        retrieved_text = match.get("_source", {}).get("Text")
        
        segments = retrieved_text.split("\n")
        for segment in segments:
            segment = segment.strip()
            if not segment:
                continue
            doc = nlp(segment)
            per_doc.extend([" ".join([token.text for token in sent]) for sent in doc.sents])
        
        spacyfied_text_by_doc.append(per_doc)
    return spacyfied_text_by_doc

In [23]:
print(len(hrte_hits))

34


In [24]:
## need to manually add the hrte hit because it has a null country in the json
hrte_matched = []
for hit in hrte_hits:
    hrte_matched.append(hit)

In [25]:
ivm_matched = matches_in_hits(ivm_hits, all_docids, countries_per_list)
print(f"ivm_matched - {len(ivm_matched)}")
ecjd_matched = matches_in_hits(ecjd_hits, all_docids, countries_per_list)
print(f"ecjd_matched - {len(ecjd_matched)}")
et_matched = matches_in_hits(et_hits, all_docids, countries_per_list)
print(f"et_matched - {len(et_matched)}")
irs_matched = matches_in_hits(irs_hits, all_docids, countries_per_list)
print(f"irs_matched - {len(irs_matched)}")
# hrte_matched = matches_in_hits(hrte_hits, all_docids, countries_per_list)
# hrte_matched = hrte_hits[0]
print(f"hrte_matched - {len(hrte_matched)}")
gttc2_matched = matches_in_hits(gttc2_hits, all_docids, countries_per_list)
print(f"gttc2_matched - {len(gttc2_matched)}")
ecji_matched = matches_in_hits(ecji_hits, all_docids, countries_per_list)
print(f"ecji_matched - {len(ecji_matched)}")
bit_matched = matches_in_hits(bit_hits, all_docids, countries_per_list)
print(f"bit_matched - {len(bit_matched)}")
wht_matched = matches_in_hits(wht_hits, all_docids, countries_per_list)
print(f"wht_matched - {len(wht_matched)}")
tns_matched = matches_in_hits(tns_hits, all_docids, countries_per_list)
print(f"tns_matched - {len(tns_matched)}")
ttcls_matched = matches_in_hits(ttcls_hits, all_docids, countries_per_list)
print(f"ttcls_matched - {len(ttcls_matched)}")


ivm_matched - 158
ecjd_matched - 834
et_matched - 336
irs_matched - 1000
hrte_matched - 34
gttc2_matched - 314
ecji_matched - 674
bit_matched - 223
wht_matched - 18
tns_matched - 222
ttcls_matched - 841


In [ ]:
matches = [
ivm_matched,
ecjd_matched,
et_matched,
irs_matched,
hrte_matched,
gttc2_matched,
ecji_matched,
bit_matched,
wht_matched, 
tns_matched, 
ttcls_matched]

In [27]:
def cropped_sample(texts, number=100):
    if len(texts) < number:
        print(f"Less docs than required -- using {len(texts)} instead")
    return texts[:number]

In [ ]:
# ivm_subset = cropped_sample(ivm_matched, number=6)
# ivm_texts = get_texts(ivm_subset)
# print("ivm done")
# ecjd_subset = cropped_sample(ecjd_matched, number=30)
# ecjd_texts = get_texts(ecjd_subset)
# print("ecjd done")
# et_subset = cropped_sample(et_matched, number=6)
# et_texts = get_texts(et_subset)
# print("et done")
# irs_subset = cropped_sample(irs_matched, number=10)
# irs_texts = get_texts(irs_subset)
# print("irs done")
# hrte_subset = cropped_sample(hrte_hits, number=8)
# hrte_texts = get_texts(hrte_subset)
# print("hrte done")
gttc2_subset = cropped_sample(gttc2_matched, number=15)
gttc2_texts = get_texts(gttc2_subset)
print("gttc2 done")
# ecji_subset = cropped_sample(ecji_matched, number=5)
# ecji_texts = get_texts(ecji_subset)
# print("ecji done")
# bit_subset = cropped_sample(bit_matched, number=5)
# bit_texts = get_texts(bit_subset)
# print("bit done")
# wht_subset = cropped_sample(wht_matched, number=2)
# wht_texts = get_texts(wht_subset)
# print("wht done")
# tns_subset = cropped_sample(tns_matched, number=80)
# tns_texts = get_texts(tns_subset)
# print("tns done")
# ttcls_subset = cropped_sample(ttcls_matched, number=60)
# ttcls_texts = get_texts(ttcls_subset)
# print("ttcls done")


ivm done
ecjd done
et done
irs done
hrte done
gttc2 done
ecji done
bit done
wht done
tns done
ttcls done


In [50]:
variables = [ivm_texts, ecjd_texts, et_texts, irs_texts, hrte_texts, gttc2_texts, ecji_texts, bit_texts, wht_texts, tns_texts, ttcls_texts]

In [51]:
# variables = [ecjd_texts]
collection_names = ["ivm", "ecjd", "et", "irs", "hrte", "gttc2", "ecji", "bit", "wht", "tns", "ttcls"]
# collection_names = ["ecjd"]
counter = 0
for name, collection in zip(collection_names, variables):
    final_sents = []
    for doc in collection:
        for sent in doc:
            final_sents.append([sent])
    
    with open(f"{name}_sentences.json", "w") as f:
        json.dump(final_sents, f)
    
    print(f"{name}: {len(final_sents)} sentences")
    counter += len(final_sents)

counter

ivm: 2300 sentences
ecjd: 6760 sentences
et: 3553 sentences
irs: 1727 sentences
hrte: 2110 sentences
gttc2: 741 sentences
ecji: 1101 sentences
bit: 1592 sentences
wht: 1142 sentences
tns: 917 sentences
ttcls: 1644 sentences


23587

In [ ]:
import random

def texts_to_sentences_json_with_boundaries(texts, output_path):
    """this saves sentences and returns document boundary indices."""
    final_sents = []
    doc_boundaries = []  # list of (start_idx, end_idx) per document
    idx = 0
    for doc in texts:
        start = idx
        for sent in doc:
            final_sents.append([sent])
            idx += 1
        doc_boundaries.append((start, idx))
    
    with open(output_path, "w") as f:
        json.dump(final_sents, f)
    
    print(f"Wrote {len(final_sents)} sentences from {len(doc_boundaries)} docs")
    return doc_boundaries

In [ ]:
# ivm_subset = cropped_sample(ivm_matched, number=2)
ivm_texts = get_texts(ivm_subset)
ivm_boundaries = texts_to_sentences_json_with_boundaries(ivm_texts, "ivm_sentences.json")
print("ivm done")
# ecjd_subset = cropped_sample(ecjd_matched, number=17)
ecjd_texts = get_texts(ecjd_subset)
ecjd_boundaries = texts_to_sentences_json_with_boundaries(ecjd_texts, "ecjd_sentences.json")
print("ecjd done")
# et_subset = cropped_sample(et_matched, number=5)
et_texts = get_texts(et_subset)
et_boundaries = texts_to_sentences_json_with_boundaries(et_texts, "et_sentences.json")
print("et done")
# irs_subset = cropped_sample(irs_matched, number=7)
irs_texts = get_texts(irs_subset)
irs_boundaries = texts_to_sentences_json_with_boundaries(irs_texts, "irs_sentences.json")
print("irs done")
# hrte_subset = cropped_sample(hrte_hits, number=4)
hrte_texts = get_texts(hrte_subset)
hrte_boundaries = texts_to_sentences_json_with_boundaries(hrte_texts, "hrte_sentences.json")
print("hrte done")
# gttc2_subset = cropped_sample(gttc2_matched, number=5)
gttc2_texts = get_texts(gttc2_subset)
gttc2_boundaries = texts_to_sentences_json_with_boundaries(gttc2_texts, "gttc2_sentences.json")
print("gttc2 done")
# ecji_subset = cropped_sample(ecji_matched, number=2)
ecji_texts = get_texts(ecji_subset)
ecji_boundaries = texts_to_sentences_json_with_boundaries(ecji_texts, "ecji_sentences.json")
print("ecji done")
# bit_subset = cropped_sample(bit_matched, number=2)
bit_texts = get_texts(bit_subset)
bit_boundaries = texts_to_sentences_json_with_boundaries(bit_texts, "bit_sentences.json")
print("bit done")
# wht_subset = cropped_sample(wht_matched, number=2)
wht_texts = get_texts(wht_subset)
wht_boundaries = texts_to_sentences_json_with_boundaries(wht_texts, "wht_sentences.json")
print("wht done")
# tns_subset = cropped_sample(tns_matched, number=16)
tns_texts = get_texts(tns_subset)
tns_boundaries = texts_to_sentences_json_with_boundaries(tns_texts, "tns_sentences.json")
print("tns done")
# ttcls_subset = cropped_sample(ttcls_matched, number=30)
ttcls_texts = get_texts(ttcls_subset)
ttcls_boundaries = texts_to_sentences_json_with_boundaries(ttcls_texts, "ttcls_sentences.json")
print("ttcls done")

collection_names = ["ivm", "ecjd", "et", "irs", "hrte", "gttc2", "ecji", "bit", "wht", "tns", "ttcls"]
variables = [ivm_texts, ecjd_texts, et_texts, irs_texts, hrte_texts, gttc2_texts, ecji_texts, bit_texts, wht_texts, tns_texts, ttcls_texts]

all_boundaries = {}
for name, texts in zip(collection_names, variables):
    boundaries = texts_to_sentences_json_with_boundaries(texts, f"{name}_sentences.json")
    all_boundaries[name] = boundaries
    
    # saving these boundaries for later
    with open(f"{name}_doc_boundaries.json", "w") as f:
        json.dump(boundaries, f)


Wrote 2300 sentences from 6 docs
ivm done
Wrote 6760 sentences from 30 docs
ecjd done
Wrote 3553 sentences from 6 docs
et done
Wrote 1727 sentences from 10 docs
irs done
Wrote 2110 sentences from 8 docs
hrte done
Wrote 741 sentences from 12 docs
gttc2 done
Wrote 1101 sentences from 5 docs
ecji done
Wrote 1592 sentences from 5 docs
bit done
Wrote 1142 sentences from 2 docs
wht done
Wrote 917 sentences from 80 docs
tns done
Wrote 1644 sentences from 60 docs
ttcls done
Wrote 2300 sentences from 6 docs
Wrote 6760 sentences from 30 docs
Wrote 3553 sentences from 6 docs
Wrote 1727 sentences from 10 docs
Wrote 2110 sentences from 8 docs
Wrote 741 sentences from 12 docs
Wrote 1101 sentences from 5 docs
Wrote 1592 sentences from 5 docs
Wrote 1142 sentences from 2 docs
Wrote 917 sentences from 80 docs
Wrote 1644 sentences from 60 docs


In [ ]:
for document in ecjd_texts:
    print(len(document))
    # for sents in document:
    #     print(len(sents))

In [ ]:
collection_names = ["ivm", "ecjd", "et", "irs", "hrte", "gttc2", "ecji", "bit", "wht", "tns", "ttcls"]
# collection_names = ["ecjd"]

for name, collection in zip(collection_names, variables):
    final_sents = []
    for doc in collection:
        for sent in doc:
            final_sents.append([sent])
    
    with open(f"{name}_sentences.json", "w") as f:
        json.dump(final_sents, f)
    
    print(f"{name}: {len(final_sents)} sentences")

# LLM labeling here

### This section uses assistance from Claude for the LLM-client functions, and to deal with API-timeout issues and for some trouble shooting

In [64]:
from dotenv import load_dotenv
import os
import asyncio
import json
import os
import random
import time
! pip install openai
from openai import AsyncAzureOpenAI, APITimeoutError, RateLimitError, AzureOpenAI
from pydantic import BaseModel
from seqeval.metrics import classification_report
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 27.5 MB/s  0:00:00
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [openai]2m4/5 [openai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-build 25.11.1 requires jsonschema>=4.19, but you have jsonschema 3.2.0 which is incompatible.
jupyterlab-server 2.28.0 requires jsonschema>=4.18.0, but you have jsonschema 3.2.0 which is incompatible.


In [55]:
load_dotenv(".env")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")

In [ ]:
def get_list_of_sentences(file_path):
    """this returns a list of lists for sentences and list of lists for labels"""
    current_sent = []
    current_label = []
    all_sents = []
    all_labels = []

    with open(file_path, "r", encoding="utf-8") as infile:
        lines = infile.readlines()
        for line in lines:
            line = line.strip()
            if line == "":
                if len(current_sent) > 0:
                    all_sents.append(current_sent)
                    all_labels.append(current_label)
                    current_sent = []
                    current_label = []
            else:
                splitted = line.split("\t")
                token_part = splitted[0]
                label_part = splitted[1]
                current_sent.append(token_part)
                current_label.append(label_part)

        if len(current_sent) > 0:
            all_sents.append(current_sent)
            all_labels.append(current_label)
    
    return all_sents, all_labels


def results_classified(predicted_file, gold_file= r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll"):
    gold_sents, gold_labels = get_list_of_sentences(gold_file)
    llm_sents, llm_labels= get_list_of_sentences(predicted_file)
    y_true, y_pred = gold_labels, llm_labels
    report = classification_report(y_true, y_pred, digits=2)
    print(report)
    return
    
    # print(len(llm_sent_2), len(llm_labels_2))
    

In [ ]:
test_file = Path(r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll")
tokens = []
labels = []
tokens_sentence = []
labels_sentence = []
token_count = []
with open(test_file, "r") as f:
    for line in f.readlines():
        if len(line) > 1:
            line = line.strip()
            token, label = line.split("\t")
            tokens_sentence.append(token)
            labels_sentence.append(label)
        else:
            if tokens_sentence:
                tokens.append(tokens_sentence)
                labels.append(labels_sentence)
                
            tokens_sentence = []
            labels_sentence = []
    if tokens_sentence:
        tokens.append(tokens_sentence)
        labels.append(labels_sentence)

print(len(tokens))

final_sents = []
counter = 0
for tok in tokens:
    new_sentence = " ".join(tok)
    final_sents.append([new_sentence])

with open("sentences.json", "w") as json_file:
    json.dump(final_sents, json_file)

656


In [ ]:
print(final_sents[0:5])

In [59]:
from enum import Enum
class Label(str, Enum):
    PERSON = 1
    ORG = 2
    GPE = 3
    LAW = 4
    DATE = 5
    TAX_TYPE = 6
    TAX_CONCEPT = 7
    PROVISION = 8
    JURISDICTION = 9
    COURT = 10

class Entity(BaseModel):
    token_text: str
    token_index: int   # key from the input token map
    label:  Label  # PER, LOC, or ORG #make e num -- to bound the values the variable can take
    entity_index: int      # the entity index so that it can be converted into B/I spans
    sent_index: int #sentence id

class NERResponse(BaseModel):
    entities: list[Entity]


In [ ]:
DOC_BOUNDARY = "###DOC###"

def ner_to_conll(token_map, entities, doc_boundaries=None):
    """this function converts a token map and entity list to conll with bio labels"""
    entity_list = []
    for thingies in entities: #convert entitiy objects into dictionaries
        sentence_dict = {
            "sentence_index": thingies.sent_index,
            "entity_info": {
                "token": thingies.token_text,
                "token_index": thingies.token_index,
                "predicted_label": str(thingies.label).split(".")[1],
                "entity_index": thingies.entity_index
            }
        }
        entity_list.append(sentence_dict)

    result = []
    prev_ent_index = None
    for dictionaries in token_map:
        prev_ent_index = None
        mapped_tokens = dictionaries["token_map"]
        sent_num = dictionaries["sentence_number"]
        #insert a doc boundary before the first sent of each doc
        if doc_boundaries:
            for start, end in doc_boundaries:
                if sent_num == start:
                    result.append(DOC_BOUNDARY)
                    break
        #add a blank line between sentences
        if result and result[-1] != DOC_BOUNDARY:
            result.append("\n")
        #process tokens in their original order    
        for i in sorted(mapped_tokens.keys()):
            token = mapped_tokens[i]
            matched = False
            for entry in entity_list: #now we check if the current tok has a prediction
                if entry["sentence_index"] == sent_num and entry["entity_info"]["token_index"] == i:
                    label = entry["entity_info"]["predicted_label"]
                    ent_idx = entry["entity_info"]["entity_index"]
                    if ent_idx == prev_ent_index: #tokens with the same entity index are considered the same predicted span and given
                        prefix = "I" #corresponding B/I- labels
                    else:
                        prefix = "B"
                        prev_ent_index = ent_idx
                    matched = True
                    result.append((token, f"{prefix}-{label}"))
                    break
            if not matched: #tokens without an ent pred are labeled O
                result.append((token, "O"))
                prev_ent_index = None
    return result

def write_shuffled_conll(results, output_path):
    """this shuffles documents and writes the results to conll"""
    import random
    random.seed(36)
    
    docs = []
    current_doc = []
    for element in results: #splitting results into individual docs
        if element == DOC_BOUNDARY:
            if current_doc:
                docs.append(current_doc)
            current_doc = []
        else:
            current_doc.append(element)
    if current_doc:
        docs.append(current_doc)
    #shuffle docs
    random.shuffle(docs)
    
    with open(output_path, "w", encoding="utf-8") as outfile: #write to conll
        for doc in docs:
            outfile.write("-DOCSTART-\tO\n\n")
            for element in doc:
                if isinstance(element, tuple):
                    t, l = element
                    outfile.write(f"{t}\t{l}\n")
                else:
                    outfile.write(element)
            outfile.write("\n")


In [61]:
def create_client() -> AsyncAzureOpenAI:
    return AsyncAzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version = "2024-12-01-preview"
    )

client = create_client()

model = "gpt-5.4-mini"
deployment ="gpt-5.4-mini"

In [62]:
SYSTEM_PROMPT = """You are a Legal and Tax NER classifier. Given a JSON object containing a sentence_number and a token_map (mapping token indices to tokens), identify all named entities and return a JSON array of entity tokens.

─── LABELS ───

PERSON
  Covers: named or specifically referenced individuals in their human capacity, including judges, advocates general, named parties, and definite singular references resolvable to one specific individual in context (e.g., Mr. Bosal, Advocate General Konll, the applicant). Spans include personal names with attached titles or roles. 
	Excludes: generic plurals (taxpayers, judges), corporate or legal persons even when named like individuals (use ORG), and unresolved generic role references (a taxpayer in this situation).

ORG
  Covers: named non-judicial organizations — companies, agencies, supranational bodies, tax authorities, professional bodies (e.g., Bosal Holding BV, European Commission, OECD, HMRC). Spans include the full proper name with name-constitutive modifiers (European in European Commission, Internal in Internal Revenue Service) and standard legal-form suffixes (BV, plc, GmbH). 
	Excludes: courts and tribunals (use COURT), jurisdictional modifiers that merely localize a generic institution, generic descriptions without naming (the company, the holding), and entity-type concepts (holding company, subsidiary) (use TAX_CONCEPT).

GPE
  Covers: specific, named geopolitical entities — countries, cities, regions, supranational unions with concrete identity (e.g., Belgium, Brussels, European Union, EU, California). Spans cover the bare name; articles are included only when part of the official name (The Hague). 
	Excludes: demonyms and adjectival forms (Belgian, Dutch, European) entirely are never tagged on their own and are stripped when they modify another entity, abstract role-based references (use JURISDICTION), and geographic regions without political identity (Europe as a continent).

LAW
  Covers: named statutory or treaty-level instruments referred to as a whole, without article or section pinpointing (e.g., TFEU, EC Treaty, Parent-Subsidiary Directive, Income Tax Act 2007). Spans include the full official title, standard acronyms, definite back-references that unambiguously resolve to a previously named instrument (the Treaty, the Directive, the Code), and years embedded in a statutory title. 
  Excludes: pinpoint references (use PROVISION), case law and judgments, and generic legal-concept terms (the law, legislation).

DATE
  Covers: absolute or relative temporal expressions denoting a point or period (e.g., 18 September 2003, 1990, 2019). Spans cover only the date expression itself; contextualizing nouns that label what kind of period it is are stripped. 
	Excludes: durations not anchored to a calendar point (for five years), vague temporal references (recently, previously, last year), and dates embedded inside another entity's name (the 2007 in Income Tax Act 2007 stays inside the LAW span).

TAX_TYPE
  Covers: named tax instruments as they appear in context (e.g., corporate income tax, withholding tax, VAT).
	Exclude: jurisdictional modifiers, generic nouns alone (tax, rate), and verbal or metaphorical uses (withholding documents, a tax on patience).

TAX_CONCEPT
  Covers: domain terms that aren't named instruments: legal doctrines, planning mechanisms, behaviors, and economic concepts (e.g., transfer pricing, arm's length, permanent establishment).
	Excludes: accounting-standard vocabulary such as GAAP, IFRS, or depreciation.

PROVISION
  Covers: specific articles, sections, paragraphs, or sub-paragraphs within a named instrument, including the host instrument when cited together (e.g., Article 49 TFEU, Article 4(1) of the Parent-Subsidiary Directive, Section 3(2), paragraph 3). Spans cover the full citation as a single entity — pinpoint plus host instrument plus subdivisions — and coordinated pinpoints sharing one host as one span (Articles 43 and 49 EC). 
	Excludes: the host instrument cited alone without a pinpoint (use LAW), case-paragraph references (paragraph 23 of the judgment), and recital references unless project-scoped in.

JURISDICTION
  Covers: abstract or role-based references to a state in its legal or fiscal capacity, where the specific country is unnamed or generalized (e.g., Member State, residence state, source state, host state, contracting state, third country). Spans include the role descriptor with role-defining qualifiers (the Member State concerned, the State of residence). 
	Excludes: named countries even when filling such a role (use GPE) and purely geographic terms without legal-role meaning.

COURT
  Covers: judicial bodies, tribunals, and adjudicative offices (e.g., Court of Justice, European Court of Justice, CJEU, Hoge Raad, Bundesfinanzhof). Spans include the full proper name with name-constitutive modifiers (European in European Court of Justice), standard acronyms, and case-specific unnamed references (the referring court, the national court). 
	Excludes: jurisdictional modifiers that merely localize a generic court, individual judges or advocates general by name (use PERSON), and metaphorical uses (the court of public opinion).

─── EXAMPLES ───

Example 1 (no entities):
Input: {"sentence_number": 0, "token_map": {"0": "The", "1": "outcome", "2": "of", "3": "this", "4": "case", "5": "was", "6": "quite", "7": "predictable", "8": "."}}
Output: []

Example 2 (PROVISION):
Input: {"sentence_number": 1, "token_map": {"0": "Residents", "1": "are", "2": "subject", "3": "to", "4": "world-wide", "5": "taxation", "6": "by", "7": "virtue", "8": "of", "9": "Article", "10": "2", "11": "of", "12": "the", "13": "LIR", "14": "."}}
Output: [
  {"sent_index": 1, "token_index": 9, "token_text": "Article", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 10, "token_text": "2", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 11, "token_text": "of", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 12, "token_text": "the", "entity_index": 0, "label": 8},
  {"sent_index": 1, "token_index": 13, "token_text": "LIR", "entity_index": 0, "label": 8}
]

Example 3 (ORG + GPE + DATE):
Input: {"sentence_number": 2, "token_map": {"0": "On", "1": "15", "2": "March", "3": "2017", "4": ",", "5": "the", "6": "European", "7": "Commission", "8": "issued", "9": "a", "10": "decision", "11": "requiring", "12": "Luxembourg", "13": "to", "14": "recover", "15": "the", "16": "unlawful", "17": "state", "18": "aid", "19": "."}}
Output: [
  {"sent_index": 2, "token_index": 1, "token_text": "15", "entity_index": 0, "label": 5},
  {"sent_index": 2, "token_index": 2, "token_text": "March", "entity_index": 0, "label": 5},
  {"sent_index": 2, "token_index": 3, "token_text": "2017", "entity_index": 0, "label": 5},
  {"sent_index": 2, "token_index": 6, "token_text": "European", "entity_index": 1, "label": 2},
  {"sent_index": 2, "token_index": 7, "token_text": "Commission", "entity_index": 1, "label": 2},
  {"sent_index": 2, "token_index": 12, "token_text": "Luxembourg", "entity_index": 2, "label": 3}
]

Example 4 (TAX_TYPE + TAX_CONCEPT + jurisdictional modifier stripped):
Input: {"sentence_number": 3, "token_map": {"0": "The", "1": "Dutch", "2": "corporate", "3": "income", "4": "tax", "5": "does", "6": "not", "7": "apply", "8": "to", "9": "profits", "10": "attributed", "11": "to", "12": "a", "13": "permanent", "14": "establishment", "15": "in", "16": "cases", "17": "involving", "18": "transfer", "19": "pricing", "20": "adjustments", "21": "."}}
Output: [
  {"sent_index": 3, "token_index": 2, "token_text": "corporate", "entity_index": 0, "label": 6},
  {"sent_index": 3, "token_index": 3, "token_text": "income", "entity_index": 0, "label": 6},
  {"sent_index": 3, "token_index": 4, "token_text": "tax", "entity_index": 0, "label": 6},
  {"sent_index": 3, "token_index": 13, "token_text": "permanent", "entity_index": 1, "label": 7},
  {"sent_index": 3, "token_index": 14, "token_text": "establishment", "entity_index": 1, "label": 7},
  {"sent_index": 3, "token_index": 18, "token_text": "transfer", "entity_index": 2, "label": 7},
  {"sent_index": 3, "token_index": 19, "token_text": "pricing", "entity_index": 2, "label": 7
]

Example 5 (COURT + PERSON + JURISDICTION + TAX_CONCEPT):
Input: {"sentence_number": 4, "token_map": {"0": "Advocate", "1": "General", "2": "Kokott", "3": "argued", "4": "before", "5": "the", "6": "Court", "7": "of", "8": "Justice", "9": "that", "10": "the", "11": "residence", "12": "state", "13": "must", "14": "grant", "15": "relief", "16": "from", "17": "double", "18": "taxation", "19": "."}}
Output: [
  {"sent_index": 4, "token_index": 0, "token_text": "Advocate", "entity_index": 0, "label": 1},
  {"sent_index": 4, "token_index": 1, "token_text": "General", "entity_index": 0, "label": 1},
  {"sent_index": 4, "token_index": 2, "token_text": "Kokott", "entity_index": 0, "label": 1},
  {"sent_index": 4, "token_index": 6, "token_text": "Court", "entity_index": 1, "label": 10},
  {"sent_index": 4, "token_index": 7, "token_text": "of", "entity_index": 1, "label": 10},
  {"sent_index": 4, "token_index": 8, "token_text": "Justice", "entity_index": 1, "label": 10},
  {"sent_index": 4, "token_index": 11, "token_text": "residence", "entity_index": 2, "label": 9},
  {"sent_index": 4, "token_index": 12, "token_text": "state", "entity_index": 2, "label": 9},
  {"sent_index": 4, "token_index": 17, "token_text": "double", "entity_index": 3, "label": 7},
  {"sent_index": 4, "token_index": 18, "token_text": "taxation", "entity_index": 3, "label": 7}
]

Example 6 (PROVISION + GPE + TAX_TYPE + JURISDICTION):
Input: {"sentence_number": 5, "token_map": {"0": "Under", "1": "Article", "2": "49", "3": "TFEU", "4": ",", "5": "Belgium", "6": "may", "7": "not", "8": "impose", "9": "withholding", "10": "tax", "11": "on", "12": "dividends", "13": "paid", "14": "to", "15": "a", "16": "parent", "17": "company", "18": "in", "19": "another", "20": "Member", "21": "State", "22": "."}}
Output: [
  {"sent_index": 5, "token_index": 1, "token_text": "Article", "entity_index": 0, "label": 8},
  {"sent_index": 5, "token_index": 2, "token_text": "49", "entity_index": 0, "label": 8},
  {"sent_index": 5, "token_index": 3, "token_text": "TFEU", "entity_index": 0, "label": 8},
  {"sent_index": 5, "token_index": 5, "token_text": "Belgium", "entity_index": 1, "label": 3},
  {"sent_index": 5, "token_index": 9, "token_text": "withholding", "entity_index": 2, "label": 6},
  {"sent_index": 5, "token_index": 10, "token_text": "tax", "entity_index": 2, "label": 6},
  {"sent_index": 5, "token_index": 20, "token_text": "Member", "entity_index": 3, "label": 9},
  {"sent_index": 5, "token_index": 21, "token_text": "State", "entity_index": 3, "label": 9}

  ─── BOUNDARY RULES ───

1. Tag the minimal named span. Jurisdictional modifiers before a TAX_TYPE are not part of the entity: in 'Dutch corporate income tax', only 'corporate income tax' is TAX_TYPE.
2. When a provision reference includes the law name, tag the entire reference as a single PROVISION.
3. Each token belongs to at most one entity.
4. Only return tokens that are part of entities. Do not return O labels.

─── OUTPUT FORMAT ───

Return ONLY a JSON array. Each element represents one entity token:
{
  "sent_index": <sentence_number from input>,
  "token_index": <token position from token_map>,
  "token_text": <exact token string>,
  "entity_index": <integer grouping multi-token entities, starting at 0>,
  "label": "<one of the following integers representing the respective label: PERSON = 1, ORG = 2, GPE = 3, LAW = 4, DATE = 5, TAX_TYPE = 6, TAX_CONCEPT = 7, PROVISION = 8, JURISDICTION = 9, COURT = 10>"
}

Consecutive tokens sharing the same entity_index form one entity span. Increment entity_index for each new entity.
"""

In [ ]:
def load_sentences(path):
    raw = json.load(open(path))
    return [group[0] for group in raw]


def build_token_map_lists(sentences):
    """this converts each sentence string into a numbered token map dict."""
    list_of_mappings = []
    for i, sentence in enumerate(sentences):
        tokens = sentence.split(" ")
        list_of_mappings.append({
            "sentence_number": i,
            "token_map": {j: token for j, token in enumerate(tokens)}})
    return list_of_mappings


async def run_ner(client, token_map):
    response = await client.beta.chat.completions.parse(
        model=deployment,
        temperature=0.0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(token_map)},
        ],
        response_format=NERResponse,
    )
    usage = {
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }
    return response.choices[0].message.parsed, usage

Defining the main function --> same code used for all the collections

In [ ]:
async def main():
    counter = 0
    sentences = load_sentences("ivm_sentences.json") #load sentences + make token index mapping
    token_maps = build_token_map_lists(sentences)
    tasks = [run_ner(client, tm) for tm in token_maps] #make one asynchronous NERC request for each sentence
    results = await asyncio.gather(*tasks) #run all NERC requests concurrently

    all_entities = []
    #keep track of token usage across the reqs
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities) #combine entities returned for all sentences
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")
    
    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities
#run NERC inference and collect predicted ents
ivm_dictionary = await main()
#save raw ent preds with their sentence and token indices
with open("raw_ivm_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in ivm_dictionary], f, indent=2)
sentences = load_sentences("ivm_sentences.json")
token_maps = build_token_map_lists(sentences) #recreate the mappings for conll conversion
with open("ivm_doc_boundaries.json", "r") as f:
    ivm_boundaries = json.load(f) #load earlier saved doc boundaries to preserve them when writing

results = ner_to_conll(token_maps, ivm_dictionary, ivm_boundaries) #write to conll
write_shuffled_conll(results, "labeled_ivm.conll") #shuffled documents


APITimeoutError: Request timed out.

: 

: 

In [ ]:
async def main():
    counter = 0
    sentences = load_sentences("bit_sentences.json")
    # sentences = sentences[0]
    token_maps = build_token_map_lists(sentences)

    # client = create_client()
    tasks = [run_ner(client, tm) for tm in token_maps]
    results = await asyncio.gather(*tasks)

    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities)
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")

    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities
bit_dictionary = await main()
with open("raw_bit_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in bit_dictionary], f, indent=2)
sentences = load_sentences("bit_sentences.json")
token_maps = build_token_map_lists(sentences)
with open("bit_doc_boundaries.json", "r") as f:
    bit_boundaries = json.load(f)
results = ner_to_conll(token_maps, bit_dictionary, bit_boundaries)
write_shuffled_conll(results, "labeled_bit.conll")

In [ ]:
async def main():
    counter = 0
    sentences = load_sentences("tns_sentences.json")
    token_maps = build_token_map_lists(sentences)
    tasks = [run_ner(client, tm) for tm in token_maps]
    results = await asyncio.gather(*tasks)

    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities)
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")

    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities
tns_dictionary = await main()
with open("raw_tns_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in tns_dictionary], f, indent=2)
sentences = load_sentences("tns_sentences.json")
token_maps = build_token_map_lists(sentences)
with open("tns_doc_boundaries.json", "r") as f:
    tns_boundaries = json.load(f)
results = ner_to_conll(token_maps, tns_dictionary, tns_boundaries)
write_shuffled_conll(results, "labeled_tns.conll")

In [ ]:
async def main():
    counter = 0
    sentences = load_sentences("ttcls_sentences.json")
    token_maps = build_token_map_lists(sentences)
    tasks = [run_ner(client, tm) for tm in token_maps]
    results = await asyncio.gather(*tasks)

    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities)
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")

    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities
ttcls_dictionary = await main()
with open("raw_ttcls_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in ttcls_dictionary], f, indent=2)
sentences = load_sentences("ttcls_sentences.json")
token_maps = build_token_map_lists(sentences)
with open("ttcls_doc_boundaries.json", "r") as f:
    ttcls_boundaries = json.load(f)
results = ner_to_conll(token_maps, ttcls_dictionary, ttcls_boundaries)
write_shuffled_conll(results, "labeled_ttcls.conll")

In [ ]:
async def main():
    counter = 0
    sentences = load_sentences("gttc2_sentences.json")
    token_maps = build_token_map_lists(sentences)
    tasks = [run_ner(client, tm) for tm in token_maps]
    results = await asyncio.gather(*tasks)

    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities)
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")

    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities
gttc2_dictionary = await main()
with open("raw_gttc2_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in gttc2_dictionary], f, indent=2)
sentences = load_sentences("gttc2_sentences.json")
token_maps = build_token_map_lists(sentences)
with open("gttc2_doc_boundaries.json", "r") as f:
    gttc2_boundaries = json.load(f)
results = ner_to_conll(token_maps, gttc2_dictionary, gttc2_boundaries)
write_shuffled_conll(results, "labeled_gttc2.conll")

In [ ]:
async def main():
    counter = 0
    sentences = load_sentences("ecji_sentences.json")
    token_maps = build_token_map_lists(sentences)
    tasks = [run_ner(client, tm) for tm in token_maps]
    results = await asyncio.gather(*tasks)

    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities)
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")

    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities
ecji_dictionary = await main()
with open("raw_ecji_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in ecji_dictionary], f, indent=2)
sentences = load_sentences("ecji_sentences.json")
token_maps = build_token_map_lists(sentences)
with open("ecji_doc_boundaries.json", "r") as f:
    ecji_boundaries = json.load(f)
results = ner_to_conll(token_maps, ecji_dictionary, ecji_boundaries)
write_shuffled_conll(results, "labeled_ecji.conll")

In [ ]:
async def main():
    counter = 0
    sentences = load_sentences("hrte_sentences.json")
    token_maps = build_token_map_lists(sentences)
    tasks = [run_ner(client, tm) for tm in token_maps]
    results = await asyncio.gather(*tasks)

    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities)
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")

    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities
hrte_dictionary = await main()
with open("raw_hrte_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in hrte_dictionary], f, indent=2)
sentences = load_sentences("hrte_sentences.json")
token_maps = build_token_map_lists(sentences)
with open("hrte_doc_boundaries.json", "r") as f:
    hrte_boundaries = json.load(f)
results = ner_to_conll(token_maps, hrte_dictionary, hrte_boundaries)
write_shuffled_conll(results, "labeled_hrte.conll")

In [ ]:
async def main():
    counter = 0
    sentences = load_sentences("irs_sentences.json")
    token_maps = build_token_map_lists(sentences)
    tasks = [run_ner(client, tm) for tm in token_maps]
    results = await asyncio.gather(*tasks)

    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities)
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")

    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities
irs_dictionary = await main()
with open("raw_irs_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in irs_dictionary], f, indent=2)
sentences = load_sentences("irs_sentences.json")
token_maps = build_token_map_lists(sentences)
with open("irs_doc_boundaries.json", "r") as f:
    irs_boundaries = json.load(f)
results = ner_to_conll(token_maps, irs_dictionary, irs_boundaries)
write_shuffled_conll(results, "labeled_irs.conll")

In [ ]:
async def main():
    counter = 0
    sentences = load_sentences("wht_sentences.json")
    token_maps = build_token_map_lists(sentences)
    tasks = [run_ner(client, tm) for tm in token_maps]
    results = await asyncio.gather(*tasks)

    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    for ner_response, usage in results:
        all_entities.extend(ner_response.entities)
        for key in total_usage:
            total_usage[key] += usage[key]

    print(f"Found {len(all_entities)} entities:")
    print(all_entities)
    for entity in all_entities:
        print(f"  [{entity.sent_index}] {entity.token_text} ({entity.label})")

    print(f"\nToken usage across {len(results)} requests:")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities
wht_dictionary = await main()
with open("raw_wht_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in wht_dictionary], f, indent=2)
sentences = load_sentences("wht_sentences.json")
token_maps = build_token_map_lists(sentences)
with open("wht_doc_boundaries.json", "r") as f:
    wht_boundaries = json.load(f)
results = ner_to_conll(token_maps, wht_dictionary, wht_boundaries)
write_shuffled_conll(results, "labeled_wht.conll")

In [ ]:
import asyncio
from openai import APITimeoutError, RateLimitError

SEMAPHORE = asyncio.Semaphore(5) #limit the number of concurrent API requests

async def run_ner_with_retry(client, token_map, max_retries=5): #run NERC with retries for time-out and rate-limit errors
    async with SEMAPHORE:
        for attempt in range(max_retries):
            try:
                return await run_ner(client, token_map)
            #retry if the api times out or rate limits the request
            except (APITimeoutError, RateLimitError) as e:
                wait = 2 ** attempt
                print(f"Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                await asyncio.sleep(wait)
        #skip request if all retry attempts fail
        print(f"Failed after {max_retries} attempts, skipping")
        return None

async def main():
    sentences = load_sentences("ecjd_sentences.json")
    token_maps = build_token_map_lists(sentences)
    total = len(token_maps)

    tasks = [run_ner_with_retry(client, tm) for tm in token_maps] #creates an asynchronous task for each sent
    
    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    completed = 0
    failed = 0
    #process reqs as they finish
    for future in asyncio.as_completed([asyncio.ensure_future(t) for t in tasks]):
        result = await future
        completed += 1
        if result is None:
            failed += 1
        else:
            ner_response, usage = result
            #collect ents returned by llm
            all_entities.extend(ner_response.entities)
            for key in total_usage:
                total_usage[key] += usage[key]
        if completed % 50 == 0 or completed == total:
            print(f"Progress: {completed}/{total} ({failed} failed)")

    print(f"\nFound {len(all_entities)} entities")
    print(f"Failed: {failed}/{total}")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities

ecjd_dictionary = await main()
with open("raw_ecjd_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in ecjd_dictionary], f, indent=2)
sentences = load_sentences("ecjd_sentences.json")
token_maps = build_token_map_lists(sentences)
with open("ecjd_doc_boundaries.json", "r") as f:
    ecjd_boundaries = json.load(f)
results = ner_to_conll(token_maps, ecjd_dictionary, ecjd_boundaries)
write_shuffled_conll(results, "labeled_ecjd.conll")

In [ ]:
SEMAPHORE = asyncio.Semaphore(5)

async def run_ner_with_retry(client, token_map, max_retries=5):
    async with SEMAPHORE:
        for attempt in range(max_retries):
            try:
                return await run_ner(client, token_map)
            except (APITimeoutError, RateLimitError) as e:
                wait = 2 ** attempt 
                print(f"Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                await asyncio.sleep(wait)
        print(f"Failed after {max_retries} attempts, skipping")
        return None

async def main():
    sentences = load_sentences("et_sentences.json")
    token_maps = build_token_map_lists(sentences)
    total = len(token_maps)

    tasks = [run_ner_with_retry(client, tm) for tm in token_maps]
    
    all_entities = []
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    completed = 0
    failed = 0

    for future in asyncio.as_completed([asyncio.ensure_future(t) for t in tasks]):
        result = await future
        completed += 1
        if result is None:
            failed += 1
        else:
            ner_response, usage = result
            all_entities.extend(ner_response.entities)
            for key in total_usage:
                total_usage[key] += usage[key]
        if completed % 50 == 0 or completed == total:
            print(f"Progress: {completed}/{total} ({failed} failed)")

    print(f"\nFound {len(all_entities)} entities")
    print(f"Failed: {failed}/{total}")
    print(f"  Prompt tokens:     {total_usage['prompt_tokens']:,}")
    print(f"  Completion tokens: {total_usage['completion_tokens']:,}")
    print(f"  Total tokens:      {total_usage['total_tokens']:,}")

    return all_entities

et_dictionary = await main()
with open("raw_et_entities.json", "w", encoding="utf-8") as f:
    json.dump([{
        "sent_index": e.sent_index,
        "token_index": e.token_index,
        "token_text": e.token_text,
        "entity_index": e.entity_index,
        "label": str(e.label).split(".")[1]
    } for e in et_dictionary], f, indent=2)
sentences = load_sentences("et_sentences.json")
token_maps = build_token_map_lists(sentences)
with open("et_doc_boundaries.json", "r") as f:
    et_boundaries = json.load(f)
results = ner_to_conll(token_maps, et_dictionary, et_boundaries)
write_shuffled_conll(results, "labeled_et.conll")

# Combining the conll files together to make LLM data splits

In [55]:
import os
collection_files = [
    "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_ivm.conll", "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_ecjd.conll", "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_et.conll",
    "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_irs.conll", "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_hrte.conll", "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_gttc2.conll",
    "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_ecji.conll", "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_bit.conll", "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_wht.conll",
    "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_tns.conll", "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/labeled_ttcls.conll"
]


In [46]:
# for filepath in collection_files:
#     with open(filepath, "r", encoding="utf-8") as f:
#         lines = f.readlines()
    
#     cleaned = []
#     for line in lines:
#         if line.strip() == "O" or line.strip() == "":
#             if line.strip() == "":
#                 cleaned.append(line)  # keep blank lines as sentence boundaries
#             continue  # skip the bare "O" lines
#         cleaned.append(line)
    
#     with open(filepath, "w", encoding="utf-8") as f:
#         f.writelines(cleaned)
    
#     print(f"Cleaned {filepath}")

In [66]:
total_docs = 0
total_sentences = 0
total_tokens = 0

for filepath in collection_files:
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read().strip()
    raw_docs = content.split("-DOCSTART-\tO")
    docs = []
    for doc in raw_docs:
        if doc.strip():
            docs.append(doc.strip())
    file_sentences = 0
    file_tokens = 0
    for doc in docs:
        sentences = doc.split("\n\n")
        for sentence in sentences:
            if sentence.strip():
                file_sentences += 1
                file_tokens += len(sentence.splitlines())
    print(
        f"{filepath[78:-6]}: "
        f"{len(docs)} docs, "
        f"{file_sentences} sentences, "
        f"{file_tokens} tokens"
    )
    total_docs += len(docs)
    total_sentences += file_sentences
    total_tokens += file_tokens

print("\nTOTAL")
print("docs:", total_docs)
print("sentences:", total_sentences)
print("tokens:", total_tokens)

labeled_ivm: 6 docs, 2300 sentences, 19540 tokens
labeled_ecjd: 30 docs, 6668 sentences, 214968 tokens
labeled_et: 6 docs, 3553 sentences, 30039 tokens
labeled_irs: 10 docs, 1727 sentences, 51868 tokens
labeled_hrte: 8 docs, 2101 sentences, 66222 tokens
labeled_gttc2: 15 docs, 888 sentences, 37314 tokens
labeled_ecji: 5 docs, 1101 sentences, 40838 tokens
labeled_bit: 5 docs, 1590 sentences, 41601 tokens
labeled_wht: 2 docs, 1142 sentences, 9468 tokens
labeled_tns: 80 docs, 917 sentences, 25924 tokens
labeled_ttcls: 60 docs, 1644 sentences, 49865 tokens

TOTAL
docs: 227
sentences: 23631
tokens: 587647


In [57]:
def read_conll_to_lists(filepath):
    all_tokens = []
    all_labels = []
    current_tokens = []
    current_labels = []

    with open(filepath, "r", encoding="utf-8") as infile:
        for line in infile:
            line = line.rstrip("\n")
            if line.strip() == "":
                if current_tokens:
                    all_tokens.append(current_tokens)
                    all_labels.append(current_labels)
                    current_tokens = []
                    current_labels = []
            else:
                parts = line.split("\t")
                if len(parts) == 2:
                    token, label = parts[0], parts[1]
                    current_tokens.append(token)
                    current_labels.append(label)
        if current_tokens:
            all_tokens.append(current_tokens)
            all_labels.append(current_labels)

    return all_tokens, all_labels

In [65]:
llm_train_docs = []
llm_dev_docs = []
import random
random.seed(23)
for filepath in collection_files:
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read().strip()
    raw_docs = content.split("-DOCSTART-\tO")
    docs = [doc.strip() for doc in raw_docs if doc.strip()]
    random.shuffle(docs)
    split = max(1, int(len(docs) * 0.90)) # making the 90-10 split
    llm_train_docs.extend(docs[:split])
    llm_dev_docs.extend(docs[split:])
    print(f"{filepath[78:-6]}: {len(docs)} docs -> {split} train, {len(docs) - split} dev")
# now we have a 90-10 train dev split for the llm docs!
## now we can write to conll!
def write_conll(docs, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for doc in docs:
            f.write(doc.strip())
            f.write("\n\n")

train_path = "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/LLM_TRAINING.conll"
dev_path = "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LLM-eval/LLM_VALIDATION.conll"

write_conll(llm_train_docs, train_path)
write_conll(llm_dev_docs, dev_path)


labeled_ivm: 6 docs -> 5 train, 1 dev
labeled_ecjd: 30 docs -> 27 train, 3 dev
labeled_et: 6 docs -> 5 train, 1 dev
labeled_irs: 10 docs -> 9 train, 1 dev
labeled_hrte: 8 docs -> 7 train, 1 dev
labeled_gttc2: 15 docs -> 13 train, 2 dev
labeled_ecji: 5 docs -> 4 train, 1 dev
labeled_bit: 5 docs -> 4 train, 1 dev
labeled_wht: 2 docs -> 1 train, 1 dev
labeled_tns: 80 docs -> 72 train, 8 dev
labeled_ttcls: 60 docs -> 54 train, 6 dev


# Making a mixed dataset (LLM data + approximate domain data)

In [59]:
random.seed(23)
def load_edgar_documents(path):
    documents = []
    current_document = []
    current_sentence = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:

            if line.strip() == "-DOCSTART-\tO":
                if current_sentence: #finish the previous sentence
                    current_document.append(current_sentence)
                    current_sentence = []
                if current_document: #finish the previous document
                    documents.append(current_document)
                current_document = []
            elif line.strip() == "": # blank line is a sentence boundary
                if current_sentence:
                    current_document.append(current_sentence)
                    current_sentence = []
            else:
                current_sentence.append(line)
        if current_sentence:
            current_document.append(current_sentence)
        if current_document:
            documents.append(current_document)

    return documents

edgar_train = load_edgar_documents("/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/EDGAR_TRAINING.conll")
edgar_val = load_edgar_documents("/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/EDGAR_VALIDATION.conll")

def edgar_doc_to_string(doc):
    sentence_strings = []

    for sentence in doc:
        sentence_text = "".join(sentence).rstrip("\n")
        sentence_strings.append(sentence_text)

    return "\n\n".join(sentence_strings)

edgar_train_docs = []

for doc in edgar_train:
    edgar_train_docs.append(edgar_doc_to_string(doc))

edgar_val_docs = []

for doc in edgar_val:
    edgar_val_docs.append(edgar_doc_to_string(doc))

In [60]:
def load_sentences(path):
    sentences = []
    current = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip() == "":
                if current:
                    sentences.append(current)
                    current = []
            else:
                current.append(line)

    if current:
        sentences.append(current)

    return sentences

indian_train_sents = load_sentences("/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/inlegal_train_FINAL.conll")
indian_val_sents = load_sentences("/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/inlegal_validation_FINAL.conll")

In [61]:
indian_train_docs = []

for sentence in indian_train_sents:
    indian_train_docs.append("".join(sentence).rstrip("\n"))

indian_val_docs = []

for sentence in indian_val_sents:
    indian_val_docs.append("".join(sentence).rstrip("\n"))

In [62]:
combined_train = (llm_train_docs + edgar_train_docs + indian_train_docs)
combined_dev = (llm_dev_docs + edgar_val_docs + indian_val_docs)
random.shuffle(combined_train)
random.shuffle(combined_dev)

In [64]:
def write_combined(path, docs):
    with open(path, "w", encoding="utf-8") as f:

        for doc in docs:
            f.write("-DOCSTART-\tO\n\n")
            f.write(doc.strip())
            f.write("\n\n")

write_combined("MIXED_TRAIN.conll", combined_train)
write_combined("MIXED_DEV.conll", combined_dev)